In [5]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load Dataset Hasil Tahap 2
file_dataset = "../data/processed/cases.csv"
folder_eval = "../data/eval"
os.makedirs(folder_eval, exist_ok=True)

if os.path.exists(file_dataset):
    df = pd.read_csv(file_dataset)
    print(f"✅ Sukses memuat dataset! Total: {len(df)} kasus waris.")
else:
    raise FileNotFoundError("❌ File cases.csv tidak ditemukan!")

# 2. Splitting Data (Rasio 80:20 sesuai silabus)
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(f" - Jumlah Data Train (Kasus Lama) : {len(df_train)}")
print(f" - Jumlah Data Test (Kasus Baru)  : {len(df_test)}")

# 3. REPRESENTASI VEKTOR 1: TF-IDF
# Kita gunakan teks utuh untuk ekstraksi fitur kata
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train['text_full'])

print(f"✅ Vektor TF-IDF Berhasil Dibuat! Ukuran matriks train: {X_train_tfidf.shape}")

✅ Sukses memuat dataset! Total: 36 kasus waris.
 - Jumlah Data Train (Kasus Lama) : 28
 - Jumlah Data Test (Kasus Baru)  : 8
✅ Vektor TF-IDF Berhasil Dibuat! Ukuran matriks train: (28, 4942)


In [4]:
!pip install torch transformers

     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ----------------------------- ---------- 30.7/41.5 kB 1.3 MB/s eta 0:00:01
     -------------------------------------- 41.5/41.5 kB 976.1 kB/s eta 0:00:00
     ---------------------------------------- 0.0/57.4 kB ? eta -:--:--
     --------------------- ------------------ 30.7/57.4 kB 1.4 MB/s eta 0:00:01
     ---------------------------------------- 57.4/57.4 kB 1.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/123.0 MB 991.0 kB/s eta 0:02:05
   ---------------------------------------- 0.1/123.0 MB 787.7 kB/s eta 0:02:37
   ---------------------------------------- 0.1/123.0 MB 939.4 kB/s eta 0:02:11
   ---------------------------------------- 0.2/123.0 MB 833.5 kB/s eta 0:02:28
   ---------------------------------------- 0.2/123.0 MB 807.1 kB/s eta 0:02:33
   ---------------------------------------- 0.2/123.0 MB 778.2 kB


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import torch
from transformers import AutoTokenizer, AutoModel

print("🔄 Sedang memuat pre-trained model IndoBERT (indobenchmark/indobert-base-p1)...")

# 1. Inisialisasi Tokenizer dan Model IndoBERT sesuai panduan silabus
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

def hitung_bert_embedding(texts):
    """Fungsi untuk mengubah list teks menjadi vektor embedding BERT"""
    embeddings = []
    # Matikan perhitungan gradien agar proses ekstraksi lebih cepat dan hemat memori
    with torch.no_grad():
        for text in texts:
            # Potong teks jika terlalu panjang (max_length=512 adalah batas standar BERT)
            inputs = tokenizer(str(text), return_tensors="pt", padding=True, truncation=True, max_length=512)
            outputs = model(**inputs)
            
            # Ambil rata-rata token (mean pooling) dari last_hidden_state sebagai representasi kalimat/dokumen
            vektor_dokumen = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
            embeddings.append(vektor_dokumen)
            
    return np.array(embeddings)

print("🔄 Memulai proses encoding teks ke BERT Embedding (ini memerlukan waktu beberapa saat)...")
# Kita ekstrak dari kolom ringkasan_fakta agar komputasi BERT tidak terlalu berat dan tetap fokus pada inti kasus
X_train_bert = hitung_bert_embedding(df_train['ringkasan_fakta'].tolist())

print(f"✅ REPRESENTASI VEKTOR 2: BERT Embedding Berhasil!")
print(f"Ukuran Matriks BERT Train: {X_train_bert.shape} (Jumlah dokumen, 768 dimensi vektor)")

c:\Users\Nabila Aurellya\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 Sedang memuat pre-trained model IndoBERT (indobenchmark/indobert-base-p1)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39759.28it/s]


🔄 Memulai proses encoding teks ke BERT Embedding (ini memerlukan waktu beberapa saat)...
✅ REPRESENTASI VEKTOR 2: BERT Embedding Berhasil!
Ukuran Matriks BERT Train: (28, 768) (Jumlah dokumen, 768 dimensi vektor)


In [11]:
from sklearn.svm import SVC
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- iii. Model Retrieval (Inisialisasi & Latih Model SVM pada TF-IDF) ---
print("🔄 Melatih Model Machine Learning (SVM) pada representasi TF-IDF...")

# Karena SVM klasifikasi butuh label/target, kita gunakan 'case_id' sebagai kelas uniknya
# Jadi SVM akan belajar mengenali ciri khas teks dari masing-masing ID Kasus
model_svm = SVC(kernel='linear', probability=True, random_state=42)
model_svm.fit(X_train_tfidf, df_train['case_id'])

print("✅ Model SVM berhasil dilatih dan siap digunakan untuk retrieval!")


# --- iv. Fungsi Retrieval dengan Dukungan Klasifikasi ML & Transformer ---
def retrieve(query: str, pendekatan: str = "tfidf_svm", k: int = 5):
    """
    Fungsi Retrieval adaptif sesuai mandat silabus UMM:
    1) 'tfidf_svm' : Menggunakan Model ML SVM (Mendukung probabilistik klasifikasi)
    2) 'bert'      : Menggunakan Model Transformer (IndoBERT Embedding + Cosine)
    """
    query_clean = query.lower()
    
    # PENDEKATAN 1: Machine Learning (SVM pada TF-IDF)
    if pendekatan == "tfidf_svm":
        # 1) Hitung vektor kueri
        query_vec = tfidf_vectorizer.transform([query_clean])
        
        # 2) Gunakan model SVM untuk menghitung probabilitas kemiripan kueri terhadap semua case_id
        probabilitas = model_svm.predict_proba(query_vec).flatten()
        
        # Ambil indeks kelas case_id berdasarkan probabilitas tertinggi
        top_k_indices = probabilitas.argsort()[::-1][:k]
        
        hasil_retrieval = []
        for idx in top_k_indices:
            cid = model_svm.classes_[idx]
            row = df_train[df_train['case_id'] == cid].iloc[0]
            hasil_retrieval.append({
                'case_id': row['case_id'],
                'no_perkara': row['no_perkara'],
                'score': round(float(probabilitas[idx]), 4), # Skor berupa probabilitas prediksi SVM
                'pasal': row['pasal'],
                'pihak': row['pihak']
            })
            
    # PENDEKATAN 2: Transformer (IndoBERT Embedding)
    elif pendekatan == "bert":
        # 1) Hitung vektor kueri lewat IndoBERT
        query_vec = hitung_bert_embedding([query_clean])
        # 2) Hitung cosine‐similarity dengan semua case vectors
        scores = cosine_similarity(query_vec, X_train_bert).flatten()
        
        top_k_indices = scores.argsort()[::-1][:k]
        
        hasil_retrieval = []
        for idx in top_k_indices:
            row = df_train.iloc[idx]
            hasil_retrieval.append({
                'case_id': row['case_id'],
                'no_perkara': row['no_perkara'],
                'score': round(float(scores[idx]), 4), # Skor berupa tingkat kemiripan Cosine
                'pasal': row['pasal'],
                'pihak': row['pihak']
            })
    else:
        raise ValueError("Pendekatan harus 'tfidf_svm' or 'bert'")
        
    return hasil_retrieval

print("✅ Fungsi retrieve() baru dengan integrasi SVM dan IndoBERT sukses dimuat!")

🔄 Melatih Model Machine Learning (SVM) pada representasi TF-IDF...
✅ Model SVM berhasil dilatih dan siap digunakan untuk retrieval!
✅ Fungsi retrieve() baru dengan integrasi SVM dan IndoBERT sukses dimuat!


c:\Users\Nabila Aurellya\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:785: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)
c:\Users\Nabila Aurellya\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [13]:
kueri_kasus_baru = "gugatan sengketa waris anak kandung berdasarkan kompilasi hukum islam"

print("--- HASIL RETRIEVAL: MODEL ML (SVM PADA TF-IDF) ---")
display(pd.DataFrame(retrieve(kueri_kasus_baru, pendekatan="tfidf_svm", k=3)))

print("\n--- HASIL RETRIEVAL: MODEL TRANSFORMER (INDOBERT EMBEDDING) ---")
display(pd.DataFrame(retrieve(kueri_kasus_baru, pendekatan="bert", k=3)))

--- HASIL RETRIEVAL: MODEL ML (SVM PADA TF-IDF) ---


,case_id,no_perkara,score,pasal,pihak
0,case_020,2255 K/PDT/2025,0.0364,HUKUM ADAT / FARAID,"L A W A N I PUTU GDE INDRA YUDHA, B VS D A N A..."
1,case_029,666 PK/PDT/2025,0.0363,PASAL 67,PEMBANDINGTERGUGAT UNTUK SELURUHNYA VS I
2,case_016,2972 K/PDT/2025,0.0362,HUKUM ADAT / FARAID,DAHULU PARA PENGGUGAT L A W A N . S VS DAHULU ...



--- HASIL RETRIEVAL: MODEL TRANSFORMER (INDOBERT EMBEDDING) ---


,case_id,no_perkara,score,pasal,pihak
0,case_012,4528 K/PDT/2025,0.3946,PASAL 834,"L A W A N MARIA LILIPALY , BERTEMPA VS MAHKAMA..."
1,case_007,4812 K/PDT/2025,0.3707,HUKUM ADAT / FARAID,"DAHULU PENGGUGAT L A W A N STEVEN, VS DAHULU ..."
2,case_035,169 K/PDT/2026,0.3691,HUKUM ADAT / FARAID,"L A W A N LANGITAN, BERTEMPAT TINGG VS MAHKAMA..."
